# ARIA FHIR R4 quick notes for GitHub sharing

This GitHub-safe notebook documents the practical ARIA/VAIS FHIR request flow without local patient IDs, hostnames, access tokens, client secrets, file paths, or saved API outputs.

It focuses on the parts that are easy to get wrong: supported Patient search parameters, document type lookup through `ValueSet/$expand`, and `DocumentReference` upload metadata.

## Configuration

Set values through environment variables or a local `.env` that is not committed.

- `ARIA_FHIR_TOKEN_URL`: OAuth token endpoint.
- `ARIA_FHIR_BASE_URL`: FHIR R4 base URL.
- `ARIA_FHIR_CLIENT_ID`: OAuth client id.
- `ARIA_FHIR_CLIENT_SECRET`: OAuth client secret.
- `ARIA_FHIR_SCOPE`: optional scope string. Empty may use the client default.
- `ARIA_TEST_PATIENT_IDENTIFIER`: optional local test patient identifier for private testing only.

Do not commit real endpoint hostnames, tokens, secrets, patient identifiers, names, birth dates, or local document paths.

In [ ]:
from __future__ import annotations

import base64
import json
import mimetypes
import os
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.parse import urlencode

import requests

try:
    from zoneinfo import ZoneInfo
except ImportError:  # pragma: no cover
    ZoneInfo = None


def load_dotenv(path: str = ".env") -> None:
    file_path = Path(path)
    if not file_path.exists():
        return
    for raw in file_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"").strip("'"))


load_dotenv()

TOKEN_URL = os.getenv("ARIA_FHIR_TOKEN_URL", "https://FHIR_TOKEN_HOST/tokenservice/connect/token")
FHIR_BASE_URL = os.getenv("ARIA_FHIR_BASE_URL", "https://FHIR_HOST/fhir/r4").rstrip("/")
CLIENT_ID = os.getenv("ARIA_FHIR_CLIENT_ID", "")
CLIENT_SECRET = os.getenv("ARIA_FHIR_CLIENT_SECRET", "")
SCOPE = os.getenv("ARIA_FHIR_SCOPE", "")
VERIFY_TLS = os.getenv("ARIA_FHIR_VERIFY_TLS", "true").lower() in {"1", "true", "yes"}

SHOW_SECRETS = False
EXECUTE_DOCUMENT_UPLOAD = False


def mask(value: str, visible: int = 8) -> str:
    if SHOW_SECRETS:
        return value
    value = str(value or "")
    if not value:
        return "<empty>"
    return value[:visible] + "...<redacted>"


def pretty(obj) -> None:
    print(json.dumps(obj, ensure_ascii=False, indent=2))

## OAuth token and FHIR GET

The flow is OAuth client credentials:

1. `POST` to the token endpoint with form fields `grant_type`, `client_id`, `client_secret`, and optional `scope`.
2. Read `access_token`, `token_type`, and `expires_in`.
3. Send FHIR requests with `Authorization: Bearer <access_token>` and `Accept: application/fhir+json`.

The token is masked in previews unless `SHOW_SECRETS = True` is set locally.

In [ ]:
def request_token() -> str:
    if not CLIENT_ID or not CLIENT_SECRET:
        raise RuntimeError("Set ARIA_FHIR_CLIENT_ID and ARIA_FHIR_CLIENT_SECRET locally before executing live requests.")

    form = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "scope": SCOPE,
    }
    pretty({
        "method": "POST",
        "url": TOKEN_URL,
        "headers": {"Content-Type": "application/x-www-form-urlencoded"},
        "form": {**form, "client_secret": "<redacted>"},
    })

    response = requests.post(
        TOKEN_URL,
        data=form,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=60,
        verify=VERIFY_TLS,
    )
    response.raise_for_status()
    body = response.json()
    token = body.get("access_token") or ""
    pretty({
        "token_type": body.get("token_type"),
        "expires_in": body.get("expires_in"),
        "access_token": mask(token, visible=16),
    })
    return token


def fhir_get(access_token: str, path: str, params: dict | list[tuple[str, str]] | None = None) -> dict:
    url = f"{FHIR_BASE_URL}/{path.lstrip('/')}"
    final_url = url + ("?" + urlencode(params, doseq=True) if params else "")
    pretty({
        "method": "GET",
        "url": final_url,
        "headers": {"Authorization": "Bearer " + mask(access_token, visible=16), "Accept": "application/fhir+json"},
    })
    response = requests.get(
        url,
        params=params,
        headers={"Authorization": f"Bearer {access_token}", "Accept": "application/fhir+json"},
        timeout=60,
        verify=VERIFY_TLS,
    )
    response.raise_for_status()
    return response.json()


def bundle_resources(payload: dict, resource_type: str | None = None) -> list[dict]:
    if payload.get("resourceType") == "Bundle":
        resources = [(entry.get("resource") or {}) for entry in payload.get("entry") or []]
    elif payload.get("resourceType"):
        resources = [payload]
    else:
        resources = []
    if resource_type:
        return [resource for resource in resources if resource.get("resourceType") == resource_type]
    return resources

## Patient search lessons

Do not assume that every generic FHIR search parameter is implemented by ARIA.

Observed practical rules:

- `Patient?name=...` may return `OperationOutcome` with `not-supported`.
- Use the server CapabilityStatement first: `GET /metadata`.
- Commonly useful Patient search parameters include `identifier`, `_id`, `family`, `given`, `birthdate`, and `name-or-identifier`.
- Broad searches like `Patient?_count=20` can return `too-costly`; provide filters.
- `Patient.name[].text` may be absent. Build display names from `name[].given[]` and `name[].family`.

In [ ]:
def patient_display_name(patient: dict) -> str:
    names = patient.get("name") or []
    if not names:
        return "<NO NAME>"
    official = next((item for item in names if item.get("use") == "official"), names[0])
    if official.get("text"):
        return str(official["text"])
    given = " ".join(str(item) for item in official.get("given") or [] if item)
    family = str(official.get("family") or "")
    return " ".join(part for part in [given, family] if part) or "<NO NAME>"


def patient_lookup_examples(test_identifier: str = "") -> list[dict]:
    examples = [
        {"label": "visible patient id", "path": "Patient", "params": {"identifier": test_identifier or "<PATIENT_IDENTIFIER>", "_count": "5"}},
        {"label": "FHIR resource id", "path": "Patient", "params": {"_id": "<PATIENT_FHIR_ID>", "_count": "5"}},
        {"label": "supported name search", "path": "Patient", "params": {"family": "<FAMILY>", "given": "<GIVEN>", "_count": "5"}},
        {"label": "broader local search", "path": "Patient", "params": {"name-or-identifier": "<TEXT>", "_count": "5"}},
        {"label": "not supported on some ARIA servers", "path": "Patient", "params": {"name": "<TEXT>", "_count": "5"}},
    ]
    return examples


pretty(patient_lookup_examples(os.getenv("ARIA_TEST_PATIENT_IDENTIFIER", "")))

## Document types

`DocumentReference.type` must use ARIA's code, display, and code system. Do not invent the code from the display text.

Typical lookup:

1. Find provider organization: `GET /Organization?type=prov&active=true`.
2. Expand document types: `GET /ValueSet/$expand?url=http://varian.com/fhir/ValueSet/documentreference-type&publisher=<Organization-id>`.
3. Use the chosen `system`, `code`, and `display` in `DocumentReference.type.coding`.

In [ ]:
DOCUMENT_TYPE_VALUESET = "http://varian.com/fhir/ValueSet/documentreference-type"
DOCUMENT_TYPE_SYSTEM = "http://varian.com/fhir/CodeSystem/DocumentReference/documentreference-type"


def document_type_expand_request(publisher_organization_id: str = "<Organization-id>") -> dict:
    return {
        "method": "GET",
        "path": "ValueSet/$expand",
        "params": {"url": DOCUMENT_TYPE_VALUESET, "publisher": publisher_organization_id},
    }


def parse_document_types(payload: dict) -> list[dict]:
    resources = bundle_resources(payload, "ValueSet") if payload.get("resourceType") == "Bundle" else [payload]
    rows = []
    for resource in resources:
        for item in (resource.get("expansion") or {}).get("contains") or []:
            if item.get("code") and item.get("display"):
                rows.append({
                    "system": item.get("system") or DOCUMENT_TYPE_SYSTEM,
                    "code": str(item.get("code")),
                    "display": str(item.get("display")),
                })
    return rows


pretty(document_type_expand_request())

## DocumentReference upload metadata

The key correction from local testing: ARIA uses `DocumentReference.category` to classify the attachment family.

- PDF/Bilder als TIF: use category code/display `TIF`.
- Word/DOCX als Patient Document: use category code/display `Patient Document`.
- `content.attachment.contentType` still carries the real MIME type such as `application/pdf`.
- TemplateName ist ARIA-Metadaten: `UPLOAD_TEMPLATE_NAME` / `documentreference-templateName` is independent from the file format.
- `UPLOAD_PREVIEW_TEXT` maps to `DocumentReference.description`; keep it empty unless you intentionally want preview/extra text. Dort koennen weiterfuehrende Infos hinterlegt werden.
- `docStatus="preliminary"` means pending / not approved.

In [ ]:
DOCUMENT_CLASS_SYSTEM = "http://varian.com/fhir/CodeSystem/DocumentReference/documentreference-class"
TEMPLATE_NAME_EXTENSION_URL = "http://varian.com/fhir/v1/StructureDefinition/documentreference-templateName"
DOCUMENT_LOCATION_EXTENSION_URL = "http://varian.com/fhir/v1/StructureDefinition/documentreference-documentLocation"
SUPERVISOR_EXTENSION_URL = "http://varian.com/fhir/v1/StructureDefinition/documentreference-supervisor"
WORD_CONTENT_TYPES = {
    "application/msword",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
}


def document_category_for_attachment(content_type: str) -> dict:
    if content_type in WORD_CONTENT_TYPES:
        return {"coding": [{"system": DOCUMENT_CLASS_SYSTEM, "code": "Patient Document", "display": "Patient Document"}]}
    return {"coding": [{"system": DOCUMENT_CLASS_SYSTEM, "code": "TIF", "display": "TIF"}]}


def fhir_datetime_now_minus_10_minutes() -> str:
    tz = ZoneInfo("Europe/Berlin") if ZoneInfo else datetime.now().astimezone().tzinfo
    value = datetime.now(tz) - timedelta(minutes=10)
    return value.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def build_document_reference_payload(
    *,
    file_path: Path,
    patient_fhir_id: str,
    doc_type: dict,
    template_name: str = "",
    preview_text: str = "",
    author_reference: dict | None = None,
    custodian_reference: dict | None = None,
    include_file_data: bool = False,
) -> dict:
    content_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    document_date = fhir_datetime_now_minus_10_minutes()
    attachment = {"contentType": content_type, "title": file_path.name, "creation": document_date}
    if include_file_data:
        attachment["data"] = base64.b64encode(file_path.read_bytes()).decode("ascii")

    payload = {
        "resourceType": "DocumentReference",
        "extension": [{"url": DOCUMENT_LOCATION_EXTENSION_URL, "valueString": "file-server"}],
        "status": "current",
        "docStatus": "preliminary",
        "type": {"coding": [{"system": doc_type["system"], "code": doc_type["code"], "display": doc_type["display"]}], "text": doc_type["display"]},
        "category": [document_category_for_attachment(content_type)],
        "subject": {"reference": f"Patient/{patient_fhir_id}"},
        "date": document_date,
        "content": [{"attachment": attachment}],
    }
    if template_name.strip():
        payload["extension"].insert(0, {"url": TEMPLATE_NAME_EXTENSION_URL, "valueString": template_name.strip()})
    if preview_text.strip():
        payload["description"] = preview_text.strip()
    if author_reference:
        payload["author"] = [dict(author_reference)]
        if author_reference.get("reference"):
            payload["extension"].append({"url": SUPERVISOR_EXTENSION_URL, "valueReference": dict(author_reference)})
    if custodian_reference:
        payload["custodian"] = dict(custodian_reference)
    return payload

## Dry-run upload example

This example intentionally uses placeholders. It does not read a real file and does not send a POST unless you replace placeholders locally and set `EXECUTE_DOCUMENT_UPLOAD = True`.

In [ ]:
UPLOAD_FILE_PATH = Path("example.pdf")
UPLOAD_PATIENT_FHIR_ID = "<Patient-FHIR-id>"
UPLOAD_DOCUMENT_TYPE = {"system": DOCUMENT_TYPE_SYSTEM, "code": "<document-type-code>", "display": "<document-type-display>"}
UPLOAD_TEMPLATE_NAME = "<optional-template-name>"
UPLOAD_PREVIEW_TEXT = ""
UPLOAD_AUTHOR_REFERENCE = {"reference": "Practitioner/<id>", "display": "<user>"}
UPLOAD_CUSTODIAN_REFERENCE = {"reference": "Organization/<id>"}

dry_run_payload = build_document_reference_payload(
    file_path=UPLOAD_FILE_PATH,
    patient_fhir_id=UPLOAD_PATIENT_FHIR_ID,
    doc_type=UPLOAD_DOCUMENT_TYPE,
    template_name=UPLOAD_TEMPLATE_NAME,
    preview_text=UPLOAD_PREVIEW_TEXT,
    author_reference=UPLOAD_AUTHOR_REFERENCE,
    custodian_reference=UPLOAD_CUSTODIAN_REFERENCE,
    include_file_data=False,
)
pretty(dry_run_payload)

if EXECUTE_DOCUMENT_UPLOAD:
    raise RuntimeError("Replace placeholders and implement the POST call locally before enabling live uploads.")